# Does `pad_stride` explain the MGLPDSNet-vs-LPDSNet gap at init?

**Hypothesis.** The two models disagree at initialisation not because of the multigrid
algorithm, but because they pad the k-space operator onto *different* grids.

`MGLPDSNet.pad_stride = s * 2**(levels - 1)`:

| config | `K` | levels | `pad_stride` |
|---|---|---|---|
| LPDSNet | `30` | 1 | **2** |
| MGLPDSNet | `[6, [4, 4, 6]]` | 3 | **8** |

Under `preproc="kspace"`, `kspace_pre_process` pads the **operator** up to that stride:
the sampling mask is nearest-neighbour resampled and the coil maps are reflect-padded.
`preprocessing/kspace.py` says what that costs:

> This is inherently approximate -- a Fourier transform on a larger grid is a different
> transform, so the padded operator is not the original one -- and it is why the configs
> size their data to a multiple of `pad_stride`, where the pad is empty and this is the
> identity.

An even-sized image is always a multiple of 2, so **LPDSNet never pads**. MGLPDSNet pads
whenever the size is not a multiple of 8 -- silently, with no error.

**Prediction.** Sweep the image size. LPDSNet should be flat. MGLPDSNet should be *equal or
better* when `size % 8 == 0` and collapse otherwise, periodically in 8. If instead both
degrade together, or MGLPDSNet is uniformly worse at every size, the hypothesis is wrong
and the problem really is the V-cycle.

Everything below is at **initialisation** -- no training, no checkpoints.

In [ ]:
import os, sys, math, json
import numpy as np
import torch
import matplotlib.pyplot as plt

# repo root: the notebook is expected to live in ImMAP/notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isfile(os.path.join(ROOT, "train.py")):
    ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, ROOT)
print("repo:", ROOT)

from models.mg_lpds import MGLPDSNet
from operators import FFT2D, Mask, Sense
from operators.noise import mri_awgn
from physics.mask import make_acc_mask
from preprocessing.kspace import kspace_pre_process

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
R = 8                 # acceleration, matches the R8 configs
ACS = 20
SIGMA = 0.005         # FIXED, not drawn -- otherwise size-to-size differences
                      # would partly be noise draws rather than the effect
print("device:", DEVICE)

## 1. Ground truth

Prefer a real preprocessed fastMRI slice (coil-combined image + its sensitivity maps).
Falls back to an analytic phantom with synthetic smooth maps so the notebook runs anywhere.
The conclusion does not depend on which one you get -- the effect is a property of the
operator, not the image -- but a real slice makes the picture panel readable.

In [ ]:
# Point this at a preprocessed volume; anything with `image` and `smaps` datasets works.
SMAP_ROOT = "../datasets/fastmri_preprocessed/brain_T2W_coil_combined/train"
SCALE_FAC = 2e3       # brain, from datasets/fastmri/loader.py::FASTMRI_PATHS
SLICE = 4


def load_real(smap_root, sl=SLICE, scale_fac=SCALE_FAC):
    import h5py, glob
    root = smap_root if os.path.isabs(smap_root) else os.path.join(ROOT, smap_root)
    files = sorted(glob.glob(os.path.join(root, "*.h5")))
    if not files:
        raise FileNotFoundError(root)
    with h5py.File(files[0], "r") as f:
        image = np.asarray(f["image"][sl:sl + 1])
        smaps = np.asarray(f["smaps"][sl:sl + 1])
    gt = torch.from_numpy(image).to(torch.complex64).reshape(1, 1, *image.shape[-2:])
    sm = torch.from_numpy(smaps).to(torch.complex64)
    sm = sm.reshape(1, -1, *sm.shape[-2:])
    print(f"real slice from {os.path.basename(files[0])}  image {tuple(gt.shape)}  "
          f"coils {sm.shape[1]}")
    return gt * scale_fac, sm


def make_phantom(H=320, W=320, NC=8, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W),
                            indexing="ij")
    mag = (0.25
           + 0.9 * torch.exp(-(xx ** 2 + yy ** 2) * 2.5)
           + 0.5 * (((xx - .30) ** 2 + (yy + .20) ** 2) < .02).float()
           + 0.4 * (((xx + .35) ** 2 + (yy - .30) ** 2) < .012).float()
           - 0.3 * (((xx + .05) ** 2 + (yy + .40) ** 2) < .008).float())
    gt = (mag * torch.exp(1j * 1.5 * xx * yy)).to(torch.complex64)[None, None]
    # smooth complex coil maps: low-order polynomials, then unit-RSS normalised
    cx = torch.cos(torch.arange(NC) * 2 * math.pi / NC)
    cy = torch.sin(torch.arange(NC) * 2 * math.pi / NC)
    sm = torch.stack([torch.exp(-((xx - a) ** 2 + (yy - b) ** 2) * 0.8)
                      * torch.exp(1j * (a * xx + b * yy) * 2.0)
                      for a, b in zip(cx, cy)]).to(torch.complex64)[None]
    sm = sm / (sm.abs().pow(2).sum(1, keepdim=True).sqrt() + 1e-8)
    print(f"analytic phantom  image (1, 1, {H}, {W})  coils {NC}")
    return gt, sm


try:
    GT_FULL, SMAPS_FULL = load_real(SMAP_ROOT)
    SOURCE = "fastMRI"
except Exception as e:
    print(f"[falling back to phantom: {type(e).__name__}: {e}]")
    GT_FULL, SMAPS_FULL = make_phantom()
    SOURCE = "phantom"

# unit-RSS check: mri_awgn's sigma only means "noise std of the coil-combined
# adjoint" when this holds, which is what makes sigma comparable across sizes.
rss = SMAPS_FULL.abs().pow(2).sum(1).sqrt()
print(f"smaps RSS: min {rss.min():.4f}  max {rss.max():.4f}  "
      f"(1.0 expected where there is coil support)")

## 2. One problem per crop size

Crop **in the image domain**, then simulate k-space from the cropped image. The forward
operator is rebuilt at the cropped size, so every size is a self-consistent, correctly
posed problem. The *only* thing that varies across the sweep is whether the size happens
to be divisible by 8.

Sigma and the mask pattern are fixed, so any difference between sizes is the operator,
not the draw.

In [ ]:
def center_crop(t, size):
    H, W = t.shape[-2:]
    h, w = size
    top, left = (H - h) // 2, (W - w) // 2
    return t[..., top:top + h, left:left + w]


def make_problem(size, gt_full=None, smaps_full=None, sigma=SIGMA, R=R, acs=ACS,
                 seed=SEED, device=DEVICE):
    # Crop to `size`, rebuild E at that size, simulate y. Returns everything the
    # networks need plus the ground truth to score against.
    gt_full = GT_FULL if gt_full is None else gt_full
    smaps_full = SMAPS_FULL if smaps_full is None else smaps_full
    hw = (size, size) if isinstance(size, int) else tuple(size)

    gt = center_crop(gt_full, hw).to(device)
    sm = center_crop(smaps_full, hw).to(device)

    mask = make_acc_mask(hw, R, acs_lines=acs, device=device)
    while mask.dim() < 4:
        mask = mask.unsqueeze(0)

    E = Mask(mask) @ FFT2D() @ Sense(sm)
    torch.manual_seed(seed)                      # same noise realisation every size
    y, _, _ = mri_awgn(gt, mask, sm, sigma, "uniform")
    sig = torch.full((1, 1, 1, 1), float(sigma), device=device)
    return dict(gt=gt, E=E, y=y, sigma=sig, size=hw)


def psnr(est, ref):
    mse = float(((est.abs() - ref.abs()) ** 2).mean())
    return 10 * math.log10(float(ref.abs().max()) ** 2 / max(mse, 1e-20))


def nrmse(est, ref):
    return float((est - ref).abs().norm() / ref.abs().norm())

## 3. The two networks, at initialisation

Built **once** and reused at every size -- the parameters do not depend on the input
shape, and building is the expensive part (`spectral_normalize` runs a power method per
layer). Same seed for both, so their level-0 dictionaries start from the same draw.

Model params are read from your generated configs so this tracks the real experiment.

In [ ]:
CFG_DIR = os.path.join(ROOT, "config", "brain", "mg")


def params_from(tag):
    with open(os.path.join(CFG_DIR, f"{tag}_R8.json")) as f:
        return json.load(f)["model"]["params"]


try:
    P_FLAT = params_from("lpdsnet")
    P_MG = params_from("mglpds")
    print("model params from config/brain/mg/*_R8.json")
except FileNotFoundError:
    BASE = dict(M=169, C=1, P=7, s=2, widen=1, degrees=1, lam0=1e-3, tau0=0.5,
                theta0=0.0, alpha0=1.0, is_complex=True, preproc="kspace",
                resize_noise=True)
    P_FLAT, P_MG = dict(BASE, K=30), dict(BASE, K=[6, [4, 4, 6]])
    print("configs not found; using the generator's defaults")

print("  LPDSNet  ", {k: P_FLAT[k] for k in ("K", "M", "P", "s", "preproc")})
print("  MGLPDSNet", {k: P_MG[k] for k in ("K", "M", "P", "s", "preproc")})

torch.manual_seed(1)
flat = MGLPDSNet(**P_FLAT).to(DEVICE).eval()
torch.manual_seed(1)
mg = MGLPDSNet(**P_MG).to(DEVICE).eval()

nparams = lambda m: sum(p.numel() for p in m.parameters())
print(f"\nLPDSNet    levels={flat.levels}  pad_stride={flat.pad_stride}  "
      f"params={nparams(flat):,}")
print(f"MGLPDSNet  levels={mg.levels}  pad_stride={mg.pad_stride}  "
      f"params={nparams(mg):,}")
print(f"\n-> sizes divisible by {flat.pad_stride} are safe for LPDSNet, "
      f"by {mg.pad_stride} for MGLPDSNet")

## 4. The mechanism, before any scores

Run the preprocessing the models actually run, and look at the grid it hands back.
If the hypothesis is right, the two models get *different-sized* surrogates at a size
that is not a multiple of 8, and identical ones when it is.

In [ ]:
S = (min(GT_FULL.shape[-2:]) // 8) * 8          # largest multiple of 8 that fits
print(f"{'size':>6}  {'model':<10}{'pad_stride':>11}{'y~ grid':>12}   padded?")
for size in (S - 4, S):
    prob = make_problem(size)
    for name, net in (("LPDSNet", flat), ("MGLPDSNet", mg)):
        y_t, E_p, _ = kspace_pre_process(prob["y"], prob["E"], net.pad_stride)
        grid = tuple(y_t.shape[-2:])
        padded = "YES  <-- operator resampled" if grid != prob["size"] else "no"
        print(f"{size:>6}  {name:<10}{net.pad_stride:>11}{str(grid):>12}   {padded}")
    print()

### What actually gets resampled

`kspace_pre_process` does four things once the size is not a multiple of `pad_stride`, and
they are not the same operation:

| object | what happens to it | in `preprocessing/kspace.py` |
|---|---|---|
| `x_adj = E^H y` | **reflect-padded** | `_pad_complex(x_adj, pad)` |
| sampling mask | **stretched** by nearest-neighbour interpolation | `pad_operator` |
| coil maps | **reflect-padded** | `pad_operator` |
| `E^H E 1` | recomputed on the padded operator | `kspace_pre_process` |

The mask is the one to watch. It is *resampled*, not padded -- `F.interpolate(..., "nearest")`
stretches the whole line pattern onto the larger grid, so a sampled column at index `j`
moves to `round(j * H_new / H_old)`. The measurement `y` was taken at the original line
positions, so after the stretch the operator claims lines the data does not have and omits
lines it does. The coil maps meanwhile are *padded*, which keeps them aligned with the
original pixels. The two halves of `E` are grown by different rules, and the mismatch is
what the plots below show.

In [ ]:
import torch.nn.functional as F
from operators.accessors import get_mask, get_sensitivity_map
from operators.padding import calc_pad_2d

BAD = S - 4          # not a multiple of 8: MGLPDSNet pads, LPDSNet does not
prob = make_problem(BAD)
E0, y0, gt0 = prob["E"], prob["y"], prob["gt"]

pad = calc_pad_2d(*gt0.shape[-2:], mg.pad_stride)
print(f"size {BAD}x{BAD}   pad_stride {mg.pad_stride}   pad (l,r,t,b) = {tuple(pad)}"
      f"   -> {BAD + pad[0] + pad[1]}x{BAD + pad[2] + pad[3]}")

# the operator as each model actually receives it
yt_flat, E_flat, par_flat = kspace_pre_process(y0, E0, flat.pad_stride)
yt_mg, E_mg, par_mg = kspace_pre_process(y0, E0, mg.pad_stride)

m0 = get_mask(E0)[0, 0].real.float().cpu()
m1 = get_mask(E_mg)[0, 0].real.float().cpu()
s0 = get_sensitivity_map(E0)[0, 0].cpu()
s1 = get_sensitivity_map(E_mg)[0, 0].cpu()

# k-space lines: which columns are sampled before vs after the stretch
col0 = (m0.sum(0) > 0).numpy()
col1 = (m1.sum(0) > 0).numpy()
idx0, idx1 = np.flatnonzero(col0), np.flatnonzero(col1)
mapped = np.unique(np.clip(np.round(idx0 * len(col1) / len(col0)).astype(int),
                           0, len(col1) - 1))
print(f"sampled columns: {len(idx0)} before  ->  {len(idx1)} after the stretch")
print(f"  columns present after but not implied by the original: "
      f"{len(np.setdiff1d(idx1, mapped))}")
print(f"  original columns with no counterpart after:            "
      f"{len(np.setdiff1d(mapped, idx1))}")

EtE1_0 = E0.gram(torch.ones_like(gt0))[0, 0].abs().cpu()
EtE1_1 = E_mg.gram(torch.ones_like(yt_mg))[0, 0].abs().cpu()

fig, ax = plt.subplots(2, 4, figsize=(15, 7.2))

ax[0, 0].imshow(m0, cmap="gray", interpolation="nearest")
ax[0, 0].set_title(f"mask, original\n{tuple(m0.shape)}", fontsize=10)
ax[0, 1].imshow(m1, cmap="gray", interpolation="nearest")
ax[0, 1].set_title(f"mask, STRETCHED (nearest)\n{tuple(m1.shape)}", fontsize=10)

# the adjoint is REFLECT-padded, not stretched -- the other half of the
# mismatch. `_pad_complex` is the repo's own complex-safe F.pad.
from preprocessing.kspace import _pad_complex
xadj0 = E0.adjoint(y0)
xadj1 = _pad_complex(xadj0, pad)
amax = float(xadj0.abs().max())
ax[0, 2].imshow(xadj0[0, 0].abs().cpu(), cmap="gray", vmin=0, vmax=amax)
ax[0, 2].set_title(f"$E^H y$, original\n{tuple(xadj0.shape[-2:])}", fontsize=10)
ax[0, 3].imshow(xadj1[0, 0].abs().cpu(), cmap="gray", vmin=0, vmax=amax)
ax[0, 3].set_title(f"$E^H y$, REFLECT-PADDED\n{tuple(xadj1.shape[-2:])}", fontsize=10)
# outline where the original pixels sit inside the padded grid
ax[0, 3].add_patch(plt.Rectangle((pad[0] - .5, pad[2] - .5), BAD, BAD,
                                 fill=False, ec="tab:red", lw=1.2, ls="--"))

ax[1, 0].imshow(s0.abs(), cmap="viridis")
ax[1, 0].set_title(f"|smaps| coil 0, original\n{tuple(s0.shape)}", fontsize=10)
ax[1, 1].imshow(s1.abs(), cmap="viridis")
ax[1, 1].set_title(f"|smaps| coil 0, REFLECT-PADDED\n{tuple(s1.shape)}", fontsize=10)
ax[1, 2].imshow(EtE1_0, cmap="magma")
ax[1, 2].set_title("$E^H E \\mathbf{1}$, original", fontsize=10)
ax[1, 3].imshow(EtE1_1, cmap="magma")
ax[1, 3].set_title("$E^H E \\mathbf{1}$, padded operator", fontsize=10)

for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"What padding does to the operator at {BAD}x{BAD}   "
             f"(white = sampled; mask is STRETCHED, everything else is PADDED)",
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

# --- 1-D line profile: the clearest view of the damage ----------------------
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.vlines(idx0, 0.55, 1.0, color="tab:blue", lw=1.2, label=f"original ({len(idx0)} lines)")
ax.vlines(idx1, 0.0, 0.45, color="tab:red", lw=1.2, label=f"stretched ({len(idx1)} lines)")
ax.set_yticks([0.22, 0.78]); ax.set_yticklabels(["stretched", "original"])
ax.set_xlabel("k-space column index")
ax.set_xlim(-1, max(len(col0), len(col1)))
ax.set_title("Sampled phase-encode lines: the data was measured on the blue "
             "positions, the padded operator assumes the red ones")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

# --- what the two models are handed, and what they produce ------------------
with torch.no_grad():
    xf, _ = flat(y0, E=E0, sigma=prob["sigma"])
    xm, _ = mg(y0, E=E0, sigma=prob["sigma"])

vmax = float(gt0.abs().max())
panels = [("ground truth", gt0[0, 0], None),
          ("$\\tilde y$ into LPDSNet", yt_flat[0, 0], None),
          ("$\\tilde y$ into MGLPDSNet", yt_mg[0, 0], None),
          ("LPDSNet out", xf[0, 0], psnr(xf, gt0)),
          ("MGLPDSNet out", xm[0, 0], psnr(xm, gt0))]

fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
for a, (title, img, p) in zip(ax, panels):
    a.imshow(img.abs().cpu().numpy(), cmap="gray", vmin=0, vmax=vmax)
    a.set_title(f"{title}\n{tuple(img.shape)}" + ("" if p is None else f"   {p:.2f} dB"),
                fontsize=10)
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"At {BAD}x{BAD}: the two models are handed different-sized surrogates",
             fontsize=12, y=1.10)
plt.tight_layout(); plt.show()

## 5. The sweep

Five sizes around `S`: two divisible by 8, three not. All are even, so LPDSNet's stride-2
padding is empty at every one of them.

In [ ]:
SIZES = [S - 8, S - 6, S - 4, S - 2, S]

rows = []
for size in SIZES:
    prob = make_problem(size)
    with torch.no_grad():
        x_flat, _ = flat(prob["y"], E=prob["E"], sigma=prob["sigma"])
        x_mg, _ = mg(prob["y"], E=prob["E"], sigma=prob["sigma"])
    x_adj = prob["E"].adjoint(prob["y"])
    rows.append(dict(size=size, mod8=size % 8,
                     zf=psnr(x_adj, prob["gt"]),
                     flat=psnr(x_flat, prob["gt"]),
                     mg=psnr(x_mg, prob["gt"]),
                     flat_nrmse=nrmse(x_flat, prob["gt"]),
                     mg_nrmse=nrmse(x_mg, prob["gt"])))

hdr = f"{'size':>6}{'%8':>5}{'zero-fill':>11}{'LPDSNet':>10}{'MGLPDSNet':>11}{'gap dB':>9}"
print(hdr); print("-" * len(hdr))
for r in rows:
    flag = "" if r["mod8"] == 0 else "   <-- MG pads"
    print(f"{r['size']:>6}{r['mod8']:>5}{r['zf']:>11.2f}{r['flat']:>10.2f}"
          f"{r['mg']:>11.2f}{r['mg'] - r['flat']:>9.2f}{flag}")

div = [r for r in rows if r["mod8"] == 0]
non = [r for r in rows if r["mod8"] != 0]
print(f"\nmean gap (MG - LPDS), size % 8 == 0 : {np.mean([r['mg']-r['flat'] for r in div]):+.2f} dB")
print(f"mean gap (MG - LPDS), size % 8 != 0 : {np.mean([r['mg']-r['flat'] for r in non]):+.2f} dB")
print(f"LPDSNet spread across all sizes     : "
      f"{max(r['flat'] for r in rows) - min(r['flat'] for r in rows):.2f} dB")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
sz = [r["size"] for r in rows]
ax.plot(sz, [r["flat"] for r in rows], "o-", label=f"LPDSNet (pad_stride {flat.pad_stride})")
ax.plot(sz, [r["mg"] for r in rows], "s-", label=f"MGLPDSNet (pad_stride {mg.pad_stride})")
ax.plot(sz, [r["zf"] for r in rows], "k:", lw=1, label="zero-filled $E^H y$")
for r in rows:
    if r["mod8"] == 0:
        ax.axvline(r["size"], color="0.85", lw=8, zorder=0)
ax.set_xlabel("image size (grey bands: divisible by 8)")
ax.set_ylabel("PSNR (dB)")
ax.set_title(f"At initialisation, {SOURCE} data, R={R}, $\\sigma$={SIGMA}")
ax.set_xticks(sz); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. What it looks like

Same networks, same weights, two sizes four pixels apart.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))
for row, size in enumerate((S, S - 4)):
    prob = make_problem(size)
    with torch.no_grad():
        xf, _ = flat(prob["y"], E=prob["E"], sigma=prob["sigma"])
        xm, _ = mg(prob["y"], E=prob["E"], sigma=prob["sigma"])
    xa = prob["E"].adjoint(prob["y"])
    vmax = float(prob["gt"].abs().max())
    panels = [("ground truth", prob["gt"], None),
              ("zero-filled", xa, psnr(xa, prob["gt"])),
              ("LPDSNet", xf, psnr(xf, prob["gt"])),
              ("MGLPDSNet", xm, psnr(xm, prob["gt"]))]
    for ax, (title, img, p) in zip(axes[row], panels):
        ax.imshow(img[0, 0].abs().cpu().numpy(), cmap="gray", vmin=0, vmax=vmax)
        lab = title if p is None else f"{title}  {p:.2f} dB"
        ax.set_title(lab, fontsize=10)
        ax.axis("off")
    tag = "divisible by 8" if size % 8 == 0 else "NOT divisible by 8"
    axes[row][0].set_ylabel(f"{size}", fontsize=11)
    axes[row][0].axis("on"); axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    axes[row][0].set_title(f"ground truth\n{size} -- {tag}", fontsize=10)
plt.tight_layout(); plt.show()

## 7. Control: is the V-cycle itself to blame?

`alpha_x` and `alpha_z` scale the coarse-grid correction. At `alpha0 = 0` the correction is
an exact no-op, so a V-cycle reduces to its fine-grid smoothing sweeps -- and
`K = [6, [4, 4, 6]]` has `6 x 4 = 24` of them.

If the multigrid machinery were responsible for the gap, killing it would close the gap.
If instead `MGLPDSNet(alpha0=0)` lands **exactly** on `LPDSNet(K=24)` at a divisible size
and still collapses at a non-divisible one, the fine path is shared and correct and the
coarse path is not the problem.

In [ ]:
torch.manual_seed(1)
mg0 = MGLPDSNet(**dict(P_MG, alpha0=0.0)).to(DEVICE).eval()
torch.manual_seed(1)
flat24 = MGLPDSNet(**dict(P_FLAT, K=24)).to(DEVICE).eval()

print(f"{'size':>6}{'%8':>5}{'MG alpha0=0':>14}{'LPDSNet K=24':>15}{'identical?':>12}")
for size in (S, S - 4):
    prob = make_problem(size)
    with torch.no_grad():
        a, _ = mg0(prob["y"], E=prob["E"], sigma=prob["sigma"])
        b, _ = flat24(prob["y"], E=prob["E"], sigma=prob["sigma"])
    same = "yes" if torch.allclose(a, b, atol=1e-5, rtol=1e-4) else "NO"
    print(f"{size:>6}{size % 8:>5}{psnr(a, prob['gt']):>14.2f}"
          f"{psnr(b, prob['gt']):>15.2f}{same:>12}")

print("\n(the two collapse onto each other only where MGLPDSNet does not pad;")
print(" at a non-divisible size they solve different problems, so they cannot match)")

## 8. Does this actually happen in your data?

The sweep above shows the failure exists. This asks whether the training set triggers it.

`crop_size` and `center_crop` are both `null` in the generated configs, so the grid the
operator is built on is whatever the preprocessed volume happens to be -- and fastMRI brain
matrix sizes vary volume to volume. Reading only the HDF5 dataset *shapes* (no pixel data),
this counts how many volumes, and how many of the slices actually sampled, land on a size
divisible by 8.

Weighting by slices matters more than by volumes: it is the fraction of training steps that
see a resampled operator. A number well below 100% also explains why the symptom would look
like instability rather than a constant offset -- MGLPDSNet gets a corrupted operator on some
slices and a correct one on others, while LPDSNet (stride 2) is fine on every even size.

In [ ]:
import glob
from collections import Counter

import h5py

# The grid the network sees is the PREPROCESSED image/smaps grid: E is built from
# `smaps`, and `crop_size` is null. The raw k-space root is consulted only to
# reproduce the loader's `acquisition` filter, which decides which volumes are in
# the dataset at all -- survey the wrong subset and the percentages are wrong.
KSPACE_ROOT = "../datasets/fastmri/brain/multicoil_train"
SURVEY_SMAP_ROOT = SMAP_ROOT          # defined in the ground-truth cell
ANATOMY = "brain"                     # "brain" -> T2 filter, "knee" -> CORPD_FBK
SLICE_RANGE = (0, 8)                  # start_slice / end_slice from the configs
MAX_FILES = None                      # set an int for a quick look


def _abs(p):
    return p if os.path.isabs(p) else os.path.normpath(os.path.join(ROOT, p))


def _acquisition(path):
    with h5py.File(path, "r") as f:
        acq = f.attrs.get("acquisition", "")
    return acq.decode() if isinstance(acq, bytes) else str(acq)


def _keeps(acq, anatomy):
    # datasets/fastmri/loader.py::_build_file_list
    return acq == "CORPD_FBK" if anatomy == "knee" else ("T2" in acq)


def survey(smap_root, kspace_root=None, anatomy=ANATOMY, slice_range=SLICE_RANGE,
           max_files=MAX_FILES):
    sroot = _abs(smap_root)
    if not os.path.isdir(sroot):
        raise FileNotFoundError(sroot)

    names = sorted(os.path.basename(p) for p in glob.glob(os.path.join(sroot, "*.h5")))
    filtered = False
    kroot = _abs(kspace_root) if kspace_root else None
    if kroot and os.path.isdir(kroot):
        keep = []
        for n in names:
            kp = os.path.join(kroot, n)
            if not os.path.exists(kp):
                continue
            try:
                if _keeps(_acquisition(kp), anatomy):
                    keep.append(n)
            except Exception:
                continue
        if keep:
            names, filtered = keep, True
    if max_files:
        names = names[:max_files]

    lo, hi = slice_range
    by_volume, by_slice, unreadable = Counter(), Counter(), []
    for n in names:
        try:
            with h5py.File(os.path.join(sroot, n), "r") as f:
                key = "image" if "image" in f else "smaps"
                shape = f[key].shape             # header only, no pixels read
        except Exception as e:
            unreadable.append((n, f"{type(e).__name__}: {e}"))
            continue
        H, W = int(shape[-2]), int(shape[-1])
        n_sl = int(shape[0]) if len(shape) >= 3 else 1
        used = max(0, min(n_sl if hi is None else hi, n_sl) - lo)
        by_volume[(H, W)] += 1
        by_slice[(H, W)] += used
    return by_volume, by_slice, unreadable, len(names), filtered


try:
    vols, slcs, unreadable, n_files, filtered = survey(SURVEY_SMAP_ROOT, KSPACE_ROOT)
except FileNotFoundError as e:
    print(f"[survey skipped: {e} not reachable from here -- run this on the cluster]")
    vols = None

if vols:
    tot_v, tot_s = sum(vols.values()), sum(slcs.values())
    ok8_v = sum(v for (H, W), v in vols.items() if H % 8 == 0 and W % 8 == 0)
    ok8_s = sum(v for (H, W), v in slcs.items() if H % 8 == 0 and W % 8 == 0)
    ok2_v = sum(v for (H, W), v in vols.items() if H % 2 == 0 and W % 2 == 0)

    src = "acquisition-filtered" if filtered else "ALL .h5 (no acquisition filter)"
    print(f"{_abs(SURVEY_SMAP_ROOT)}\n  {tot_v} volumes ({src}), "
          f"{tot_s} slices in range {SLICE_RANGE}\n")

    print(f"  {'size':>12}{'volumes':>9}{'slices':>8}   %8   %2")
    for (H, W), v in sorted(vols.items(), key=lambda kv: -kv[1]):
        mark = "  ok" if (H % 8 == 0 and W % 8 == 0) else "  PAD"
        print(f"  {f'{H}x{W}':>12}{v:>9}{slcs.get((H, W), 0):>8}"
              f"{mark:>5}{'  ok' if (H % 2 == 0 and W % 2 == 0) else '  PAD':>6}")

    print(f"\n  MGLPDSNet (pad_stride 8): {ok8_v}/{tot_v} volumes "
          f"({100*ok8_v/max(tot_v,1):.1f}%), {ok8_s}/{tot_s} slices "
          f"({100*ok8_s/max(tot_s,1):.1f}%) need no padding")
    print(f"  LPDSNet   (pad_stride 2): {ok2_v}/{tot_v} volumes "
          f"({100*ok2_v/max(tot_v,1):.1f}%) need no padding")
    print(f"\n  -> {100*(tot_s-ok8_s)/max(tot_s,1):.1f}% of sampled slices hand "
          f"MGLPDSNet a resampled operator")

    smallest = min(min(H, W) for (H, W) in vols)
    safe = (smallest // 8) * 8
    print(f"\n  smallest dimension anywhere: {smallest}"
          f"  ->  largest multiple of 8 that fits every volume: {safe}")
    if 320 <= smallest:
        print(f"  -> set crop_size / center_crop to 320 (fits, and is a multiple of 8)")
    else:
        print(f"  -> set crop_size / center_crop to {safe}; 320 does NOT fit "
              f"every volume")
    if unreadable:
        print(f"\n  unreadable: {len(unreadable)}")
        for n, why in unreadable[:5]:
            print(f"    {n}: {why}")

## 9. Reading the result

**Hypothesis confirmed** if: LPDSNet is flat across all five sizes, MGLPDSNet matches or
beats it at the two divisible sizes and drops sharply at the three others, and the
`alpha0=0` control lands on `LPDSNet(K=24)` at the divisible size.

**Hypothesis refuted** if: MGLPDSNet is uniformly worse at every size including the
divisible ones, or both models degrade together. Then the padding is incidental and the
V-cycle is the thing to look at -- start with `alpha0` (the configs use `1.0`, while
`models/multigrid.py::VCycle` defaults to `1e-1`) and with the FAS corrections in
`PDObjectiveDownsample`.

### If confirmed, the fix

1. Set `crop_size` (or `center_crop`) to a multiple of 8 in
   `scripts/make_mg_recon_configs.py`. The configs currently use `crop_size: null`, i.e.
   full FOV, and fastMRI brain matrix sizes vary volume to volume -- so this fires on some
   slices and not others, which reads as instability rather than as a bug.
2. Make the k-space path loud. `preproc="identity"` already calls `_check_grid` and raises;
   `preproc="kspace"` pads silently. A warn-once naming the model and size would have made
   this immediate.
3. Re-check the other pair before trusting any grid result: `altsplit`'s denoiser is
   `K=6` (1 level, `pad_stride` 2) and `mgaltsplit`'s is `K=[1, [4, 4, 6]]` (3 levels,
   `pad_stride` 8), and `LPDS_DENOISER` strips `preproc` so both inherit the
   `"kspace"` default. The same asymmetry is available there.

Until sizes are a multiple of 8 for every model in the grid, flat-vs-multigrid comparisons
are confounded: the two arms are not solving the same problem.

## 10. A fix that keeps the operator exact: image-domain embedding

Padding the operator is the wrong lever. The measurement is fine; only the *network* wants a
grid divisible by `pad_stride`. So move the padding to the image domain and leave `E` alone:

$$y \;=\; M F S \cdot \mathrm{Truncate}(x')\,, \qquad x' \in \mathbb{C}^{H' \times W'},
\quad H' = \lceil H/8 \rceil \cdot 8 \;\ge\; H$$

`Truncate` crops the padded grid down to the measured one, and its adjoint is **zero-padding**.
Crop and zero-pad are an exact adjoint pair, so `E' = E @ Truncate` is a genuine linear
operator with

$$E'^H E' \;=\; T^H E^H E\, T$$

holding exactly -- no resampled mask, no reflect-padded coil maps, no approximation anywhere.
`y`, the mask and the maps are all untouched; the network simply solves for a slightly larger
image and the read-out crops it back.

This is reasonable for MRI specifically because the anatomy already sits inside the FOV, so
the pixels being added are genuinely zero signal rather than invented content.

Two things worth checking rather than assuming:

1. **Does the composition still hit the fused SENSE Gram?** `_match_sense_gram` keys on the
   first three operators and passes the rest through as an untyped tail -- the same slot
   `galerkin` uses for its grid transfers -- so `Mask @ FFT2D @ Sense @ Truncate` should match
   and cost nothing extra.
2. **The border is unconstrained.** `T` annihilates it, so the data term's gradient there is
   exactly zero: those pixels are driven only by the dictionary term `B z`, and `A` reads them
   back into `z`. They are cropped away at read-out, but they can still spend capacity and
   feed back during the unroll. The last cell measures how much energy ends up there.

In [ ]:
import torch.nn.functional as F
from operators.base import Operator


class Truncate(Operator):
    """Centred crop `big -> small`, with ZERO-PADDING as its exact adjoint.

    forward : x' on the padded grid  ->  x on the measured grid
    adjoint : x                      ->  x' with zeros outside
    """

    def __init__(self, big, small):
        self.big, self.small = tuple(big), tuple(small)
        self.top = (self.big[0] - self.small[0]) // 2
        self.left = (self.big[1] - self.small[1]) // 2

    def forward(self, x):
        t, l = self.top, self.left
        return x[..., t:t + self.small[0], l:l + self.small[1]]

    def adjoint(self, x):
        dh, dw = self.big[0] - self.small[0], self.big[1] - self.small[1]
        p = (self.left, dw - self.left, self.top, dh - self.top)
        # F.pad's reflect/constant kernels are real-only; mirror the repo's
        # _pad_complex and pad the two halves.
        if torch.is_complex(x):
            return torch.complex(F.pad(x.real, p), F.pad(x.imag, p))
        return F.pad(x, p)

    def __repr__(self):
        return f"Truncate({self.big}->{self.small})"


def next_mult(n, m):
    return ((n + m - 1) // m) * m


def embed(E, hw, stride):
    """`E @ Truncate` onto the next multiple of `stride`, plus the operator."""
    big = (next_mult(hw[0], stride), next_mult(hw[1], stride))
    T = Truncate(big, tuple(hw))
    return E @ T, T, big


# ---- correctness, before any reconstruction -------------------------------
BAD = S - 4 if (S - 4) % mg.pad_stride else S - 2      # a size that pads today
prob = make_problem(BAD)
E0, y0, gt0 = prob["E"], prob["y"], prob["gt"]
E_emb, T, big = embed(E0, (BAD, BAD), mg.pad_stride)

print(f"measured {BAD}x{BAD}  ->  embedded {big[0]}x{big[1]}   {E_emb!r}")
print(f"fused SENSE gram still matched: {E_emb._fused_gram is not None}")

torch.manual_seed(3)
xb = torch.randn(1, 1, *big, dtype=torch.complex64, device=DEVICE)
yr = torch.randn_like(y0)
lhs = torch.vdot(E_emb(xb).flatten(), yr.flatten())
rhs = torch.vdot(xb.flatten(), E_emb.adjoint(yr).flatten())
print(f"\nadjointness  <E'x, y> vs <x, E'^H y>: "
      f"relative error {abs(lhs - rhs) / abs(lhs):.3e}")

g_fast, g_ref = E_emb.gram(xb), T.adjoint(E0.gram(T.forward(xb)))
print(f"gram == T^H (E^H E) T: {torch.allclose(g_fast, g_ref, atol=1e-5)}"
      f"   max|diff| {float((g_fast - g_ref).abs().max()):.2e}")

yt_ref, _, par_ref = kspace_pre_process(y0, E0, 1)          # stride 1: no pad
yt_emb, _, par_emb = kspace_pre_process(y0, E_emb, mg.pad_stride)
print(f"\ny~ grid: reference {tuple(yt_ref.shape[-2:])}  "
      f"embedded {tuple(yt_emb.shape[-2:])}   operator pad applied: {par_emb[1]}")
print(f"mu identical to the unpadded problem: "
      f"{torch.allclose(par_ref[0], par_emb[0], atol=1e-6)}")
print(f"y~_embedded == zeropad(y~_reference): "
      f"{torch.allclose(yt_emb, T.adjoint(yt_ref), atol=1e-5)}")

### Does it flatten the sweep?

Same five sizes as section 5, now routed through the embedding. LPDSNet should be unchanged
(it never padded); MGLPDSNet should stop caring about divisibility entirely.

In [ ]:
rows_emb = []
for size in SIZES:
    prob = make_problem(size)
    E0, y0, gt0, sig = prob["E"], prob["y"], prob["gt"], prob["sigma"]

    with torch.no_grad():                                   # today's route
        xf_pad, _ = flat(y0, E=E0, sigma=sig)
        xm_pad, _ = mg(y0, E=E0, sigma=sig)

    Ef, Tf, _ = embed(E0, (size, size), flat.pad_stride)
    Em, Tm, _ = embed(E0, (size, size), mg.pad_stride)
    with torch.no_grad():                                   # embedded route
        xf_emb, _ = flat(y0, E=Ef, sigma=sig)
        xm_emb, _ = mg(y0, E=Em, sigma=sig)
    xf_emb, xm_emb = Tf.forward(xf_emb), Tm.forward(xm_emb)

    rows_emb.append(dict(size=size, mod=size % mg.pad_stride,
                         fp=psnr(xf_pad, gt0), mp=psnr(xm_pad, gt0),
                         fe=psnr(xf_emb, gt0), me=psnr(xm_emb, gt0)))

hdr = (f"{'size':>6}{'%ps':>5}   |{'LPDS pad':>10}{'MG pad':>9}{'gap':>8}"
       f"   |{'LPDS emb':>10}{'MG emb':>9}{'gap':>8}")
print(hdr); print("-" * len(hdr))
for r in rows_emb:
    print(f"{r['size']:>6}{r['mod']:>5}   |{r['fp']:>10.2f}{r['mp']:>9.2f}"
          f"{r['mp']-r['fp']:>8.2f}   |{r['fe']:>10.2f}{r['me']:>9.2f}"
          f"{r['me']-r['fe']:>8.2f}")

spread = lambda k: max(r[k] for r in rows_emb) - min(r[k] for r in rows_emb)
print(f"\nMGLPDSNet spread across sizes:  padded {spread('mp'):.2f} dB"
      f"   ->  embedded {spread('me'):.2f} dB")
print(f"mean gap (MG - LPDS):           padded "
      f"{np.mean([r['mp']-r['fp'] for r in rows_emb]):+.2f} dB"
      f"   ->  embedded {np.mean([r['me']-r['fe'] for r in rows_emb]):+.2f} dB")

### The unconstrained border

`T` annihilates the border, so `grad_x` from the data term is exactly zero there and those
pixels move only through `B z`. They are cropped at read-out, but if a large fraction of the
output energy lands outside the measured window it means the dictionary is spending capacity
on invented content -- and `A` feeds it back into `z` on the next sweep. Worth watching before
adopting this at 320x320, where the border is a much smaller fraction of the grid than in
these small test cases.

In [ ]:
prob = make_problem(BAD)
E0, y0, gt0, sig = prob["E"], prob["y"], prob["gt"], prob["sigma"]
Em, Tm, big = embed(E0, (BAD, BAD), mg.pad_stride)
with torch.no_grad():
    raw, _ = mg(y0, E=Em, sigma=sig)          # full H'xW', before the crop
    xm_pad, _ = mg(y0, E=E0, sigma=sig)       # today's route, for reference
inner = Tm.forward(raw)

tot = float(raw.abs().pow(2).sum())
ins = float(inner.abs().pow(2).sum())
area = 100 * (1 - (BAD * BAD) / (big[0] * big[1]))
print(f"border is {area:.1f}% of the {big[0]}x{big[1]} grid and holds "
      f"{100*(tot-ins)/tot:.2f}% of the output energy")

fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
vmax = float(gt0.abs().max())
panels = [("ground truth", gt0[0, 0], None),
          (f"MGLPDSNet, operator padding", xm_pad[0, 0], psnr(xm_pad, gt0)),
          (f"MGLPDSNet, embedded (raw {big[0]})", raw[0, 0], None),
          ("embedded, cropped back", inner[0, 0], psnr(inner, gt0)),
          ("|error| of the cropped result", (inner - gt0)[0, 0], None)]
for a, (title, img, p) in zip(ax, panels):
    a.imshow(img.abs().cpu().numpy(), cmap="gray", vmin=0,
             vmax=vmax if "error" not in title else vmax * 0.25)
    a.set_title(f"{title}\n{tuple(img.shape)}"
                + ("" if p is None else f"   {p:.2f} dB"), fontsize=10)
    a.set_xticks([]); a.set_yticks([])
# outline the measured window inside the raw embedded output
ax[2].add_patch(plt.Rectangle((Tm.left - .5, Tm.top - .5), BAD, BAD,
                              fill=False, ec="tab:red", lw=1.2, ls="--"))
fig.suptitle(f"Image-domain embedding at {BAD}x{BAD}: the operator is exact, "
             f"only the image grid grows", fontsize=12, y=1.10)
plt.tight_layout(); plt.show()